In [1]:
import pandas as pd
import numpy as np
import statsmodels.api as sm

df = pd.read_csv("../../Data/Monday_Combined_dataset.csv")
df["date"] = pd.to_datetime(df["date"])
df = df.sort_values("date")

# First differences
df["d_gas_austin"] = df["gas_price"].diff()
df["d_gas_us"] = df["national_gas_price"].diff()
df["d_wti"] = df["wti_price"].diff()

df = df.dropna().copy()

In [2]:
def run_adl(df, y_col, x_col, p, q):
    temp = df.copy()

    for i in range(1, p + 1):
        temp[f"{y_col}_lag{i}"] = temp[y_col].shift(i)

    for i in range(0, q + 1):
        temp[f"{x_col}_lag{i}"] = temp[x_col].shift(i)

    temp = temp.dropna()

    y = temp[y_col]

    X_cols = []

    if p > 0:
        X_cols += [f"{y_col}_lag{i}" for i in range(1, p + 1)]

    X_cols += [f"{x_col}_lag{i}" for i in range(0, q + 1)]

    X = sm.add_constant(temp[X_cols])

    model = sm.OLS(y, X).fit()

    rmse = np.sqrt((model.resid ** 2).mean())

    return {
        "p": p,
        "q": q,
        "AIC": model.aic,
        "BIC": model.bic,
        "RMSE": rmse
    }

In [3]:
results_us = []

for p in range(0, 5):
    for q in range(0, 5):
        try:
            res = run_adl(df, "d_gas_us", "d_wti", p, q)
            results_us.append(res)
        except Exception as e:
            print(f"Error at p={p}, q={q}: {e}")

results_us_df = pd.DataFrame(results_us).sort_values("BIC")
print(results_us_df.head(25))

    p  q         AIC         BIC      RMSE
6   1  1 -821.188364 -806.945638  0.049118
11  2  1 -818.048382 -800.264242  0.048924
7   1  2 -816.777802 -798.993662  0.049044
12  2  2 -817.611032 -796.270063  0.048777
16  3  1 -815.728700 -794.410943  0.048652
8   1  3 -812.784519 -791.466762  0.048930
17  3  2 -814.501536 -789.630819  0.048579
13  2  3 -812.289806 -787.419089  0.048788
21  4  1 -810.239146 -785.395614  0.048679
18  3  3 -812.660347 -784.236670  0.048564
9   1  4 -806.900515 -782.056982  0.048996
22  4  2 -808.860806 -780.468197  0.048620
14  2  4 -806.628879 -778.236270  0.048831
23  4  3 -807.149548 -775.207864  0.048592
19  3  4 -806.987473 -775.045788  0.048608
24  4  4 -805.314558 -769.823797  0.048577
3   0  3 -785.139901 -767.375103  0.051824
2   0  2 -780.486294 -766.258982  0.052807
4   0  4 -780.186799 -758.892343  0.051811
1   0  1 -749.708743 -739.026698  0.056573
5   1  0 -748.123781 -737.441736  0.056746
10  2  0 -742.570191 -728.342879  0.056817
15  3  0 -7

In [5]:
results_us = []

for p in range(0, 5):
    for q in range(0, 5):
        try:
            res = run_adl(df, "d_gas_austin", "d_wti", p, q)
            results_us.append(res)
        except Exception as e:
            print(f"Error at p={p}, q={q}: {e}")

results_us_df = pd.DataFrame(results_us).sort_values("BIC")
print(results_us_df.head(25))

    p  q         AIC         BIC      RMSE
12  2  2 -453.544810 -432.203842  0.098502
7   1  2 -448.006752 -430.222612  0.099946
8   1  3 -447.227654 -425.909897  0.099369
13  2  3 -449.267990 -424.397273  0.098594
17  3  2 -449.045673 -424.174956  0.098637
9   1  4 -444.598087 -419.754554  0.099147
14  2  4 -447.423337 -419.030728  0.098220
18  3  3 -447.268443 -418.844766  0.098594
22  4  2 -445.907287 -417.514678  0.098511
19  3  4 -446.037970 -414.096285  0.098103
23  4  3 -444.131116 -412.189432  0.098468
24  4  4 -444.038039 -408.547278  0.098103
11  2  1 -423.656660 -405.872520  0.104757
16  3  1 -419.190600 -397.872842  0.104918
6   1  1 -405.792258 -391.549531  0.109187
21  4  1 -416.112469 -391.268936  0.104797
2   0  2 -382.146330 -367.919018  0.113936
1   0  1 -377.291885 -366.609840  0.115783
3   0  3 -377.885824 -360.121026  0.114103
4   0  4 -374.630781 -353.336324  0.114048
10  2  0 -341.701622 -327.474310  0.123189
15  3  0 -337.883590 -320.118792  0.123300
5   1  0 -3